In [21]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 20)

In [22]:
data = {
    'PH' : 6.07,
    'EC': 0.18,
    'OC': 0.823529412,
    'N':61.02352941 ,
    'P':234.4716,
    'K':267.3138 ,
    'S':3.24 ,
    'Zn':6.232 ,
    'B':0.220464 ,
    'Fe':53.1 ,
    'Mn':12.336 ,
    'Cu':2.548
}


In [23]:
def normalize_more_is_better(value, xmin, xmax):
    """Higher value -> higher score. Used for OC, N, P, K, S, Zn, B, Fe, Mn, Cu."""
    score = (value - xmin) / (xmax - xmin)
    return score.clip(0, 1)          # cap between 0 and 1 — a value far above Xmax doesn't mean "extra fertile"

def normalize_less_is_better(value, xmin, xmax):
    """Lower value -> higher score. Used for EC."""
    score = (xmax - value) / (xmax - xmin)
    return score.clip(0, 1)

def normalize_optimal_range(value, optimal, max_deviation):
    """Closer to the optimal value -> higher score. Used for pH."""
    score = 1 - np.abs(value - optimal) / max_deviation
    return score.clip(0, 1)

In [24]:
BOUNDS = {
    'OC': (0.0, 1.5),      # %
    'N':  (0.0, 111.15),   # kg/ha  (= 1.5 * 74.1, since N is derived from OC)
    'P':  (0.0, 56.0),     # kg/ha
    'K':  (0.0, 336.0),    # kg/ha
    'S':  (0.0, 20.0),     # ppm
    'Zn': (0.0, 1.2),      # ppm
    'B':  (0.0, 1.0),      # ppm
    'Fe': (0.0, 10.0),     # ppm
    'Mn': (0.0, 5.0),      # ppm
    'Cu': (0.0, 1.0),      # ppm
}

EC_MIN, EC_MAX = 0.0, 2.0        # dS/m — EC uses "less is better"
PH_OPTIMAL, PH_MAX_DEVIATION = 6.5, 3.5 # pH uses "optimal range"

In [25]:
norm = pd.Series(data)
norm

PH      6.070000
EC      0.180000
OC      0.823529
N      61.023529
P     234.471600
K     267.313800
S       3.240000
Zn      6.232000
B       0.220464
Fe     53.100000
Mn     12.336000
Cu      2.548000
dtype: float64

In [26]:
norm["PH"] = normalize_optimal_range(norm["PH"],PH_OPTIMAL,PH_MAX_DEVIATION) 
norm["PH"]

0.8771428571428572

In [27]:
norm["EC"] = normalize_less_is_better(norm["EC"],EC_MIN,EC_MAX) 
norm["EC"]

0.91

In [29]:
for param, (xmin, xmax) in BOUNDS.items():
    norm[param] = normalize_more_is_better(norm[param], xmin, xmax)
    print(f"{param}: {norm[param]}")

OC: 0.36601307199999994
N: 0.004939447663763027
P: 0.017857142857142856
K: 0.002367788052721089
S: 0.0081
Zn: 0.8333333333333334
B: 0.220464
Fe: 0.1
Mn: 0.2
Cu: 1.0
